# 03 — Module selection and region responsibility

**Objective.** Convert the 13 raw RIT result sources into a union of selected factor-combination modules and compute HVR and triplet region-by-module responsibility.

RIT outputs are called **interactions** until source-specific selection. Objects that pass selection are then called **modules**. **Outputs** are written under `WORK_ROOT/03_responsibility`.

## Configuration and imports

In [ ]:
import gc
import os
from pathlib import Path

import pandas as pd

from _tutorial_utils import (
    compare_dataframe_checkpoint,
    ensure_crane_grn_importable,
    require_path,
)

ensure_crane_grn_importable()
from crane_grn.modules import (
    build_factor_by_module,
    build_union_region_by_factor,
    collect_region_union,
    compute_selection_idf,
    load_irf_results,
    select_modules_by_source,
    union_selected_modules,
)
from crane_grn.responsibility import compute_responsibility

DATA_ROOT = Path(os.environ.get("CRANE_GRN_DATA_ROOT", "../../CRANE_GRN_data")).expanduser().resolve()
WORK_ROOT = Path(os.environ.get("CRANE_GRN_TUTORIAL_WORK", "./results")).expanduser().resolve()
PBMC_INPUT_ROOT = DATA_ROOT / "PBMC" / "prepared_inputs"
PBMC_REFERENCE_ROOT = DATA_ROOT / "PBMC" / "outputs"
IRF_INPUT_ROOT = WORK_ROOT / "01_irf_inputs"
IRF_RESULT_ROOT = WORK_ROOT / "02_irf"
OUTPUT_ROOT = WORK_ROOT / "03_responsibility"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Load grouped inputs and raw RIT results

The iRF input summary preserves the biological group label, while each deterministic run directory provides feature names, raw stability scores, and extracted interaction statistics.

In [ ]:
input_summary_path = require_path(IRF_INPUT_ROOT / "irf_input_summary.tsv", "iRF input summary")
input_summary = pd.read_csv(input_summary_path, sep="\t")
input_summary = input_summary.loc[input_summary["path"].notna()].copy()
if input_summary.shape[0] != 13:
    raise ValueError("Expected 13 PBMC iRF input records.")

def runner_directory_name(input_path):
    name = Path(input_path).stem
    if name.endswith("_iRF_input"):
        name = name[:-len("_iRF_input")]
    if name.lower().startswith("pbmc_"):
        name = name[len("pbmc_"):]
    return name

input_paths = [require_path(path, "grouped iRF input") for path in input_summary["path"]]
runs = []
for row in input_summary.itertuples(index=False):
    run_name = runner_directory_name(row.path)
    run_dir = require_path(IRF_RESULT_ROOT / run_name, "iRF/RIT result directory")
    runs.append((run_name, str(row.group), run_dir))

input_region_summary, union_regions = collect_region_union(input_paths)
raw_interactions = load_irf_results(runs)
print("Union regions:", len(union_regions))
print("Raw RIT interaction rows:", raw_interactions.shape[0])
display(input_region_summary)

## Source-specific selection and module union

Interactions are ranked independently within each source and order. The paper selection uses order 8 or lower, the top 10% per order, equal IDF/stability/tree weighting, and a stability threshold of 0.5. A one-group dataset supplies one run to the same APIs.

In [ ]:
hvr_factors = pd.read_parquet(require_path(PBMC_INPUT_ROOT / "region_by_factor_hvr.parquet", "frozen HVR factor matrix"))
union_hvr_factors = build_union_region_by_factor(hvr_factors, union_regions)
selection_idf = compute_selection_idf(union_hvr_factors)
selection_options = {
    "max_order": 8,
    "top_fraction_per_order": 0.10,
    "selection_idf_weight": 0.5,
    "stability_weight": 0.5,
    "tree_weight": 0.5,
    "stability_threshold": 0.5,
}
selected_long, scored_long, selection_summary = select_modules_by_source(
    raw_interactions, selection_idf, **selection_options
)
module_catalog = union_selected_modules(selected_long)
factor_by_module = build_factor_by_module(module_catalog)

raw_interactions.to_csv(OUTPUT_ROOT / "raw_rit_interactions.tsv", sep="\t", index=False)
selected_long.to_csv(OUTPUT_ROOT / "selected_modules_by_source.tsv", sep="\t", index=False)
selection_summary.to_csv(OUTPUT_ROOT / "selection_summary.tsv", sep="\t", index=False)
module_catalog.to_csv(OUTPUT_ROOT / "module_catalog.tsv", sep="\t", index=False)
factor_by_module.to_parquet(OUTPUT_ROOT / "factor_by_module.parquet")
display(selection_summary)
print("Factor-by-module:", factor_by_module.shape)

## Inspect responsibility parameter selection

The manuscript parameter set was selected using a diagnostic grid search on the HVR region-by-factor matrix. 
This section demonstrates that selection procedure. Users interested only in manuscript-matched reproduction may skip it because the selected values are fixed explicitly in the subsequent **Compute responsibility** section.

- `tversky_alpha` and `tversky_beta = 1 - tversky_alpha` control the asymmetric factor-set similarity.
- `order_strength` controls the module-order correction.
- `top_m` is the maximum number of modules retained per region; `0` disables this sparsification during grid evaluation.
- `popularity_strength` corrects for modules that broadly match many regions.
- `softmax_temperature` controls the sharpness of the masked softmax.

The grid is evaluated only with the HVR matrix; the selected manuscript parameters are subsequently applied to both HVR and triplet responsibility calculations.

In [ ]:
from crane_grn.responsibility import (
    DEFAULT_PARAMETER_GATES,
    DEFAULT_POPULARITY_STRENGTH,
    DEFAULT_SOFTMAX_TEMPERATURE,
    parameter_grid_search,
    select_responsibility_parameters,
)

# Parameter ranges used for the PBMC manuscript analysis.
TVERSKY_GRID = tuple(i / 10 for i in range(1, 10))
TOP_M_GRID = (0, 3, 4, 5, 6, 7, 8, 9, 10)
ORDER_STRENGTH_GRID = tuple(i / 10 for i in range(0, 6))

# The public implementation fixes these two parameters during this grid search.
RESPONSIBILITY_GATES = dict(DEFAULT_PARAMETER_GATES)

display(
    pd.DataFrame(
        [{
            "popularity_strength": DEFAULT_POPULARITY_STRENGTH,
            "softmax_temperature": DEFAULT_SOFTMAX_TEMPERATURE,
            "max_stability_jsd": RESPONSIBILITY_GATES["max_stability_jsd"],
            "max_abs_order_correlation": RESPONSIBILITY_GATES["max_order_corr"],
        }]
    )
)

grid_results = parameter_grid_search(
    region_by_factor=union_hvr_factors,
    factor_by_module=factor_by_module,
    tversky_grid=TVERSKY_GRID,
    top_m_grid=TOP_M_GRID,
    order_grid=ORDER_STRENGTH_GRID,
    gates=RESPONSIBILITY_GATES,
)

# beta is determined by alpha rather than searched independently.
grid_results["tversky_beta"] = 1.0 - grid_results["tversky_alpha"]

grid_preview = (
    grid_results.loc[
        :,
        [
            "tversky_alpha",
            "tversky_beta",
            "order_strength",
            "top_m",
            "stability_jsd",
            "gini",
            "gap_median",
            "order_corr",
            "passes_absolute",
        ],
    ]
    .rename(
        columns={
            "gini": "assignment_gini",
            "gap_median": "median_assignment_gap",
            "order_corr": "order_assignment_correlation",
        }
    )
    .sort_values(
        [
            "passes_absolute",
            "stability_jsd",
            "assignment_gini",
            "median_assignment_gap",
        ],
        ascending=[False, True, True, False],
    )
)

display(grid_preview.head(12))

print(f"Grid configurations evaluated: {len(grid_results):,}")
print(
    "Configurations passing the absolute gates:",
    int(grid_results["passes_absolute"].sum()),
)

,popularity_strength,softmax_temperature,max_stability_jsd,max_abs_order_correlation
0,0.2,0.1,0.05,0.2


### Inspect the selected parameter set

The grid search balances assignment stability, sparsity, separation between the leading assignments, and dependence on module order. Lower `stability_jsd` and `assignment_gini`, higher `median_assignment_gap`, and an `order_assignment_correlation` closer to zero are preferred.

Selection follows the original two-stage procedure:

1. Retain configurations with `stability_jsd ≤ 0.05` and absolute order correlation `≤ 0.20`.
2. Within those configurations, retain results at or below the 30th percentile of assignment Gini and at or above the 70th percentile of median assignment gap.
3. Select the candidate with the best aggregate diagnostic rank.

In [ ]:
selected_parameters, relative_gates, ranked_candidates = (
    select_responsibility_parameters(
        grid_results,
        gates=RESPONSIBILITY_GATES,
    )
)

selected_parameter_table = pd.DataFrame(
    [{
        "tversky_alpha": selected_parameters["tversky_alpha"],
        "tversky_beta": 1.0 - selected_parameters["tversky_alpha"],
        "order_strength": selected_parameters["order_strength"],
        "top_m": int(selected_parameters["top_m"]),
        "popularity_strength": DEFAULT_POPULARITY_STRENGTH,
        "softmax_temperature": DEFAULT_SOFTMAX_TEMPERATURE,
        "stability_jsd": selected_parameters["stability_jsd"],
        "assignment_gini": selected_parameters["gini"],
        "median_assignment_gap": selected_parameters["gap_median"],
        "order_assignment_correlation": selected_parameters["order_corr"],
        "passes_absolute": bool(selected_parameters["passes_absolute"]),
        "selected": True,
    }]
)

display(selected_parameter_table)

selection_gate_table = pd.DataFrame(
    [{
        "max_stability_jsd": RESPONSIBILITY_GATES["max_stability_jsd"],
        "max_abs_order_correlation": RESPONSIBILITY_GATES["max_order_corr"],
        "max_assignment_gini_stage_2": relative_gates["max_gini"],
        "min_median_assignment_gap_stage_2": relative_gates["min_gap"],
    }]
)

display(selection_gate_table)

ranked_display = ranked_candidates.loc[
    :,
    [
        "tversky_alpha",
        "top_m",
        "order_strength",
        "stability_jsd",
        "gini",
        "gap_median",
        "order_corr",
        "rank_sum",
    ],
].head(10).copy()

ranked_display["tversky_beta"] = 1.0 - ranked_display["tversky_alpha"]
ranked_display["eligible"] = True
ranked_display["selected"] = (
    ranked_display.index == ranked_candidates.index[0]
)

ranked_display = ranked_display.rename(
    columns={
        "gini": "assignment_gini",
        "gap_median": "median_assignment_gap",
        "order_corr": "order_assignment_correlation",
    }
)

display(
    ranked_display[
        [
            "tversky_alpha",
            "tversky_beta",
            "order_strength",
            "top_m",
            "stability_jsd",
            "assignment_gini",
            "median_assignment_gap",
            "order_assignment_correlation",
            "rank_sum",
            "eligible",
            "selected",
        ]
    ].reset_index(drop=True)
)

### Manuscript-matched parameter confirmation

For the PBMC manuscript configuration, the selection output should confirm:

- `tversky_alpha = 0.9`
- `tversky_beta = 0.1`
- `order_strength = 0.4`
- `top_m = 3`
- `popularity_strength = 0.2`
- `softmax_temperature = 0.1`

The table above is generated from the actual selection helper rather than from hard-coded tuning output. The subsequent canonical computation should continue to declare these manuscript values explicitly and should not depend on `selected_parameters`; it can therefore run even when this optional tuning section is skipped.

## Compute responsibility

Responsibility combines IDF-weighted Tversky similarity, an order correction, a popularity correction, masked softmax, and top-three sparsification. `tversky_alpha` and `tversky_beta` describe asymmetric set similarity; they are unrelated to the later `posterior_exponent`.

In [ ]:
TVERSKY_ALPHA = 0.9
TVERSKY_BETA = 0.1
ORDER_STRENGTH = 0.4
TOP_M = 3
POPULARITY_STRENGTH = 0.2
SOFTMAX_TEMPERATURE = 0.1
assert abs(TVERSKY_ALPHA + TVERSKY_BETA - 1.0) < 1e-12

responsibility_hvr = compute_responsibility(
    region_by_factor=union_hvr_factors,
    factor_by_module=factor_by_module,
    tversky_alpha=TVERSKY_ALPHA,
    order_strength=ORDER_STRENGTH,
    top_m=TOP_M,
    popularity_strength=POPULARITY_STRENGTH,
    softmax_temperature=SOFTMAX_TEMPERATURE,
)
triplet_factors = pd.read_parquet(require_path(PBMC_INPUT_ROOT / "region_by_factor_triplet.parquet", "frozen triplet factor matrix"))
responsibility_triplet = compute_responsibility(
    region_by_factor=triplet_factors,
    factor_by_module=factor_by_module,
    tversky_alpha=TVERSKY_ALPHA,
    order_strength=ORDER_STRENGTH,
    top_m=TOP_M,
    popularity_strength=POPULARITY_STRENGTH,
    softmax_temperature=SOFTMAX_TEMPERATURE,
)
responsibility_hvr.to_parquet(OUTPUT_ROOT / "responsibility_hvr.parquet")
responsibility_triplet.to_parquet(OUTPUT_ROOT / "responsibility_triplet.parquet")
print("HVR responsibility:", responsibility_hvr.shape)
print("Triplet responsibility:", responsibility_triplet.shape)

## Interpretation, outputs, and next step

Each responsibility row distributes a region over at most three selected modules after the paper corrections. The HVR matrix supports ATAC likelihood calculation; the triplet matrix connects modules to the SCENIC+ regulatory region universe.

**Outputs:** `factor_by_module.parquet`, `responsibility_hvr.parquet`, and `responsibility_triplet.parquet` under `WORK_ROOT/03_responsibility`. **Next step:** integrate RNA and ATAC evidence in notebook 04.